# Camera Traps in New Places - Kaggle notebook
**Right panel -> Session options:** Accelerator = **GPU (T4 x2 or P100)**, Internet = **On**. Add your code dataset via *Add Input*.
Run the cells top to bottom. `/kaggle/working` is erased when the session ends, so re-run cells 1-3 in a new session (download takes ~10 min).
To keep results: *Save Version* and download from the Output tab, or run cell 8.

In [ ]:
# 1. GPU + code (the project zip you uploaded as a Kaggle Dataset)
import os, glob, shutil, zipfile
!nvidia-smi | head -12
hits = glob.glob('/kaggle/input/**/src/run_adaptation.py', recursive=True)
if hits:
    CODE = os.path.dirname(os.path.dirname(hits[0]))
else:                                   # dataset kept the zip un-extracted
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'Add your code dataset: right panel -> Add Input -> Your datasets'
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/_code')
    CODE = os.path.dirname(os.path.dirname(glob.glob('/kaggle/working/_code/**/src/run_adaptation.py', recursive=True)[0]))
W = '/kaggle/working/iwildcam_adapt'
shutil.rmtree(W, ignore_errors=True)
shutil.copytree(CODE, W, copy_function=shutil.copyfile)
os.system(f'chmod -R u+w {W}')
%cd {W}
!pip install -q timm gdown gradio

In [ ]:
# 2. Get data from Google Drive (Drive links must be shared "Anyone with the link")
import gdown
CACHE_LINK   = 'PASTE_DRIVE_FOLDER_LINK_iwildcam_cache'
CKPT_LINK    = 'PASTE_DRIVE_FOLDER_LINK_ckpt'
RESULTS_LINK = 'PASTE_DRIVE_FILE_LINK_results.zip'
D = '/kaggle/working/data'
os.makedirs(D, exist_ok=True)
gdown.download_folder(CACHE_LINK, output=f'{D}/cache', quiet=True)
gdown.download_folder(CKPT_LINK, output=f'{D}/ckpt', quiet=True)
gdown.download(RESULTS_LINK, f'{D}/results.zip', fuzzy=True, quiet=True)
!unzip -o -q {D}/results.zip -d {D}
CACHE, CKPT, RES = f'{D}/cache', f'{D}/ckpt', f'{D}/results'
!ls -la {CACHE} {CKPT}
!du -sh {D}

In [ ]:
# 3. Check that the model arrived intact: numbers must match what you got on Colab
!python -m src.baseline_eval --cache-dir {CACHE} --ckpt-dir {CKPT} --out /kaggle/working/check
import json
a = json.load(open(f'{RES}/baseline.json')); b = json.load(open('/kaggle/working/check/baseline.json'))
for bb in a:
    print(bb, '| Colab unseen acc', round(a[bb]['test']['acc'], 4), '| Kaggle', round(b[bb]['test']['acc'], 4))

In [ ]:
# 4. (optional) BN with shrinkage towards source statistics, on the test cameras (needs GPU)
for pr in [64, 256, 1024]:
    !python -m src.run_adaptation --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES} --backbones resnet50 --methods bn --sizes 16 32 64 128 --seeds 0 1 2 3 4 --bn-prior {pr}

In [ ]:
# 5. Tables + figures (uses every result file in RES, old and new)
!python -m src.analysis --results {RES} --cache-dir {CACHE} --focus-n 64
from IPython.display import Image, display
for f in ['fig_seen_unseen', 'fig_curve', 'fig_per_camera', 'fig_conditions', 'fig_empty_frac']:
    p = f'{RES}/{f}.png'
    if os.path.exists(p): display(Image(p))
!cat {RES}/summary.md

In [ ]:
# 6. (optional) a SECOND training seed -> shows how much the baseline varies (~40 min on GPU)
for bb in ['resnet50', 'dino_vits16']:
    !python -m src.train --backbone {bb} --cache-dir {CACHE} --out /kaggle/working/ckpt_seed1 --epochs 5 --seed 1
!python -m src.baseline_eval --cache-dir {CACHE} --ckpt-dir /kaggle/working/ckpt_seed1 --out /kaggle/working/results_seed1
!cat /kaggle/working/results_seed1/baseline.json

In [ ]:
# 7. Web app with a public link (Internet must be On). Do not run this when using "Save & Run All".
!python -m src.demo --cache-dir {CACHE} --ckpt-dir {CKPT} --share

In [ ]:
# 8. Pack the outputs so you can download them (Save Version -> Output tab)
!cd /kaggle/working && zip -qr results_out.zip data/results check results_seed1 2>/dev/null; ls -la /kaggle/working/results_out.zip